# Setup

## Import

In [ ]:
# !pip install bertopic
!pip install reverse_geocoder
!pip install langchain-openai
!pip install bert_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 33.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for reverse_geocoder: filename=reverse_geocoder-1.5.1-py3-none-any.whl size=2268067 sha256=25348a60e88cc3ea48c7de47b1e816a89bc30ae860e3416a85414eae2eec18a2
  Stored in directory: /root/.cache/pip/wheels/11/e1/67/6e47f0ad41ea1843d37e1fbe79c6074744a1f4aace641cf800
Successfully built reverse_geocoder
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 kB 22.8 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.19
    Uninstalling langchain-core-1.2.19:
      Successfully uninstalled langchain-core-1.2.19
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.3 MB/s eta 0:00:00


In [ ]:
import cuml.accel
cuml.accel.install()

In [ ]:
# Import libraries
import os
import pandas as pd
import numpy as np
import nltk
import reverse_geocoder as rg
import time
from sklearn.feature_extraction.text import CountVectorizer
from google.colab import drive
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer, util
from bert_score import score as calculate_bertscore
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate

# from bertopic import BERTopic
# from bertopic.dimensionality import BaseDimensionalityReduction
# from bertopic.cluster import BaseCluster
# from geopy.geocoders import Nominatim

In [ ]:
# Connection to Drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/CMS_PRO-Explainable_Clustering/'
PROCESSED_DATA_DIR = DATA_DIR + 'data/processed'

# Download stopwords for keyword extraction
nltk.download('stopwords')

Mounted at /content/drive


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
# Load data parquet
df = pd.read_parquet(PROCESSED_DATA_DIR + '/clustered_data.parquet')
df.head()

,title,description,keyword,geometry_wkt,temporal_startDate,temporal_endDate,text,centroid_lon,centroid_lat,cluster_id
3,Berichterstattung Artikel 4/5 der EG-Hochwasse...,Die Bundesanstalt für Gewässerkunde nimmt im A...,HWRM-RL. FD. Flood Directive. Floods Directive...,"POLYGON ((5.87 55.06, 5.87 47.26, 15.05 47.26,...",2012-03-06 00:00:00+00:00,2012-03-06 00:00:00+00:00,Berichterstattung Artikel 4/5 der EG-Hochwasse...,9.542000,51.940000,17
4,Natürlicher Saldo,Differenz Geburten - Sterbefälle je 1.000 Einw...,Sozialstruktur. Bevölkerung. Natürlicher Saldo...,"POLYGON ((5.11 55.11, 5.11 46.99, 15.69 46.99,...",2023-01-01 00:00:00+00:00,2023-12-31 00:00:00+00:00,Natürlicher Saldo | Differenz Geburten - Sterb...,9.342000,51.862000,6
10,Moorkundlich - ökohydrologisches Gutachten übe...,Im Zusammenhang mit der Erweiterung der Naturs...,Lebensräume und Biotope. Bewirtschaftungsgebie...,"POLYGON ((13.3692550344182 53.7189306780194, 1...",1993-01-20 00:00:00+00:00,1993-01-20 00:00:00+00:00,Moorkundlich - ökohydrologisches Gutachten übe...,13.389488,53.709726,6
15,Holzbacher Straße,"Vorhabenbezogener Bebauungsplan ""Holzbacher St...",2f23ed6e-0186-4c2b-8380-748389036509. Lokal. B...,"POLYGON ((7.51635924957007 49.9805208816045, 7...",2009-04-29 00:00:00+00:00,2014-02-21 00:00:00+00:00,Holzbacher Straße | Vorhabenbezogener Bebauung...,7.517514,49.979743,6
16,"Bebauungsplan Nr. 5 ""Entlang der Kreisstraße 1...",Die Gemeinde Renkenberge ist eine Mitgliedsgem...,Bauleitplanung. Bebauungsplan. Stadtplanung. S...,"POLYGON ((7.3784948030498 52.908738040868, 7.3...",1993-05-31 00:00:00+00:00,1993-05-31 00:00:00+00:00,"Bebauungsplan Nr. 5 ""Entlang der Kreisstraße 1...",7.379336,52.908154,6


In [ ]:
def time_method(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()
        print(f"Method {func.__name__} took {end - start:.4f}s")
        return result
    return wrapper

## Explainer

In [ ]:
# Explainer class definition
class ClusterExplainer:
    def __init__(self, df: pd.DataFrame):
        self.df = df[df.cluster_id != -1].copy()
        self.summary_df = pd.DataFrame()
        self.df['combined_text'] = (
            self.df['title'].fillna('') + '. ' +
            self.df['description'].fillna('') + '. ' +
            self.df['keyword'].fillna('') + '. '
        ).str.strip()

    def get_combined_stopwords(self):
        german_stopwords = set(stopwords.words('german'))
        english_stopwords = set(stopwords.words('english'))

        metadata_noise = {
            'id', 'basis', 'info', '000', 'ab', 'id', 'available',
            'data', 'datum', 'beschreibung', 'tmap', 'cmip6', 'wcrp'
        }

        combined_stopwords = german_stopwords | english_stopwords | metadata_noise
        return list(combined_stopwords)

    def _extract_topical(self, top_n=10):
        """Apply c-tf-idf: CountVectorizer + formula"""
        docs_per_cluster = self.df.groupby('cluster_id')['combined_text'] \
            .apply(lambda x: ' '.join(x)).reset_index()

        stopwords = self.get_combined_stopwords()

        vectorizer = CountVectorizer(
            stop_words=stopwords,
            max_features=10000,
            min_df=2,
            token_pattern=r'(?u)\b[a-zA-ZäöüßÄÖÜ]{3,}\b')

        term_frequency_matrix = vectorizer.fit_transform(
            docs_per_cluster['combined_text']).toarray()

        global_word_frequency = term_frequency_matrix.sum(axis=0)
        global_word_frequency = np.where(
            global_word_frequency == 0, 1, global_word_frequency)

        total_words_each_cluster = term_frequency_matrix.sum(axis=1)

        average_word_per_cluster = total_words_each_cluster.mean()

        tf_normalized = (term_frequency_matrix.T / (total_words_each_cluster + 1e-9)).T

        c_tf_idf = tf_normalized * \
        np.log(1 + (average_word_per_cluster / global_word_frequency))

        feature_names = vectorizer.get_feature_names_out()

        cluster_keywords = {}
        for i, row in enumerate(c_tf_idf):
            cluster_id = docs_per_cluster.iloc[i].cluster_id
            top_indices = row.argsort()[-top_n:][::-1]
            cluster_keywords[cluster_id] = [
                feature_names[idx] for idx in top_indices]

        return cluster_keywords

    # Spatial: reverse_geocoder
    def _get_spatial_label(self, cluster):
        lat = cluster.centroid_lat.median()
        lon = cluster.centroid_lon.median()

        geo_info = rg.search((lat, lon), verbose=False)[0]
        region = f"{geo_info.get('admin1', 'Unknown')}, {geo_info.get('cc', '')}"

        return region

    # Form sentence: GenAI
    def fit_explanations(self, top_n=15):
        print("Extracting keywords and location...")
        all_keywords = self._extract_topical(top_n=top_n)

        summaries = []
        for cid in sorted(self.df.cluster_id.unique()):
            cluster = self.df[self.df.cluster_id == cid]
            keywords = all_keywords.get(cid, [])

            region = self._get_spatial_label(cluster)

            year_start = int(pd.to_datetime(cluster.temporal_startDate).dt.year.quantile(0.1))
            year_end = int(pd.to_datetime(cluster.temporal_endDate).dt.year.quantile(0.9))

            summaries.append({
                'cluster_id': cid,
                # 'explanation': sentence,
                'size': len(cluster),
                'keywords': keywords,
                'region': region,
                'period': f"{year_start}-{year_end}"
            })

        self.summary_df = pd.DataFrame(summaries)
        print("Extraction complete.")

    # Visualisation (optional): plotly scatter

    # Explaination generator
    @time_method
    def generate_explainations(self, model_name):
        print(f"Generating explanations using {model_name}...")
        os.environ["OPENAI_API_KEY"] = "YOUR API TOKEN HERE"

        llm = init_chat_model(temperature=0, model=model_name)

        explanation_template = """
        You are an explainer that explain a cluster in one sentence.
        I have a cluster of datasets with the following metadata:
        - Region: {region}
        - Period: {period}
        - Cluster Size: {size}
        - Top Keywords: {keywords}
        Region is only the median location the datasets are centered around, not exact.
        USER: Please tell me what the cluster is about, when and where in one English sentence.
        ASSISTANT:
        """

        prompt = PromptTemplate(
            input_variables=["region", "period", "size", "keywords"],
            template=explanation_template
        )

        chain = prompt | llm

        gen_expl = []
        for _, cluster in self.summary_df.iterrows():
            res = chain.invoke({
                "region": cluster['region'],
                "period": cluster['period'],
                "size": cluster['size'],
                "keywords": ', '.join(cluster['keywords'])
            })
            gen_expl.append(res.content)

        self.summary_df['explanation'] = gen_expl
        print("Generation complete.")
        return self.summary_df

    def evaluate_results(self):
        """
        Evaluate the topic keywords against the content of clusters
        Evaluate the generated explanations against the topic keywords
        """
        print("Evaluating...")

        # Model selected by semantic search score
        # https://www.sbert.net/docs/sentence_transformer/pretrained_models.html#original-models
        st_model = SentenceTransformer('multi-qa-mpnet-base-dot-v1')
        kw_fidelity_scores = []

        # Keyword fidelity (cosine similarity)
        for _, cluster in self.summary_df.iterrows():
            cid = cluster['cluster_id']

            # Extract test sample
            cluster_docs = self.df[self.df.cluster_id == cid]['combined_text'].tolist()
            sample_docs = cluster_docs[:100]

            # Encode the sample
            doc_embeddings = st_model.encode(sample_docs)
            cluster_centroid = np.mean(doc_embeddings, axis=0)

            # Encode the keywords
            keyword_embeddings = st_model.encode(cluster['keywords'])
            keyword_mean_vec = np.mean(keyword_embeddings, axis=0)

            # Calculate cosine similarity
            fidelity = util.cos_sim(cluster_centroid, keyword_mean_vec).item()
            kw_fidelity_scores.append(fidelity)

        # Explanation faithfulness (Bertscore)
        candidates = self.summary_df['explanation'].tolist()
        refs = self.summary_df['keywords'].apply(lambda x: ", ".join(x)).tolist()

        precision, recall, f1 = calculate_bertscore(
            candidates,
            refs,
            model_type='xlm-roberta-base')

        self.summary_df['kw_fidelity'] = kw_fidelity_scores
        self.summary_df['exp_faithfulness'] = f1.tolist()

        print("Evaluation complete.")
        return self.summary_df

    def save_explanations(self, path):
        print("Saving explanations...")
        self.summary_df.to_excel(path, index=False)
        print("Saving complete.")

# Test Zone

gpt-5.4-(mini | pro | nano)

gpt-4.1

gpt-4o-mini

In [ ]:
# expl_dir = PROCESSED_DATA_DIR + '/Explanations/gpt-5-mini.xlsx'
# explainer_expl_eval = ClusterExplainer(df)
# explainer_expl_eval.fit_explanations(top_n=15)
# explainer_expl_eval.generate_explainations(model_name='gpt-5-mini')
# explainer_expl_eval_res = explainer_expl_eval.evaluate_results()
# explainer_expl_eval_res

In [ ]:
# explainer_expl_eval.save_explanations(path=expl_dir)

In [ ]:
expl_dir_gpt5_mini = PROCESSED_DATA_DIR + '/Explanations/gpt-5.4-mini.xlsx'
explainer_gpt5_mini = ClusterExplainer(df)
explainer_gpt5_mini.fit_explanations(top_n=15)
explainer_gpt5_mini.generate_explainations(model_name='gpt-5.4-mini')
explainer_gpt5_mini_eval = explainer_gpt5_mini.evaluate_results()
explainer_gpt5_mini_eval

Extracting keywords and location...
Extraction complete.
Generating explanations using gpt-5.4-mini...
Generation complete.
Method generate_explainations took 21.0473s
Evaluating...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluation complete.


,cluster_id,size,keywords,region,period,explanation,kw_fidelity,exp_faithfulness
0,0,2110,"[bsrn, station, monitoring, atmosphere, violet...","Oklahoma, US",1997-2017,This cluster appears to be about atmospheric a...,0.642745,0.820063
1,1,2857,"[dendrochronological, object, historical, tree...","Overijssel, NL",2003-2011,This cluster appears to be about historical de...,0.606389,0.817454
2,2,2479,"[xbt, profiles, woce, oceanography, cruise, wa...","Saint George, BM",1990-1997,This cluster appears to be mainly about oceano...,0.591898,0.836709
3,3,3815,"[nathaniel, palmer, jgofs, ctd, revelle, roger...","Tasmania, AU",1997-2000,This cluster appears to be about marine and oc...,0.645258,0.839266
4,4,3755,"[censor, bottle, chile, odp, physical, zooplan...","Ica, PE",1995-2004,This cluster is about oceanography and marine ...,0.676345,0.820188
5,5,3289,"[meteor, geob, sediment, type, core, corer, gr...","Karas, NA",1992-2000,This cluster appears to be about marine and ge...,0.698308,0.843066
6,6,23952,"[bebauungsplan, inspireidentifiziert, bodennut...","North Rhine-Westphalia, DE",1993-2023,This cluster consists of open local planning a...,0.710504,0.801803
7,7,4365,"[radiosonde, ant, polarstern, cruise, xxix, zo...",", GS",1995-2014,This cluster appears to be about **polar ocean...,0.515756,0.756883
8,8,5783,"[ice, ant, neumayer, polarstern, antarctic, an...",", FK",1992-2018,This cluster appears to be about Antarctic and...,0.668194,0.836240
9,9,2092,"[queen, ark, arctic, awi, land, expedition, se...","Sakha, RU",1993-2014,This cluster is about Arctic and subarctic ear...,0.667301,0.838908


In [ ]:
explainer_gpt5_mini.save_explanations(path=expl_dir_gpt5_mini)

Saving explanations...
Saving complete.


In [ ]:
expl_dir_gpt5 = PROCESSED_DATA_DIR + '/Explanations/gpt-5.4.xlsx'
explainer_gpt5 = ClusterExplainer(df)
explainer_gpt5.fit_explanations(top_n=15)
explainer_gpt5.generate_explainations(model_name='gpt-5.4')
explainer_gpt5_eval = explainer_gpt5.evaluate_results()
explainer_gpt5_eval

Extracting keywords and location...
Extraction complete.
Generating explanations using gpt-5.4...
Generation complete.
Method generate_explainations took 28.7925s
Evaluating...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluation complete.


,cluster_id,size,keywords,region,period,explanation,kw_fidelity,exp_faithfulness
0,0,2110,"[bsrn, station, monitoring, atmosphere, violet...","Oklahoma, US",1997-2017,This cluster appears to cover atmospheric and ...,0.642745,0.826905
1,1,2857,"[dendrochronological, object, historical, tree...","Overijssel, NL",2003-2011,This cluster is mainly about dendrochronologic...,0.606389,0.821526
2,2,2479,"[xbt, profiles, woce, oceanography, cruise, wa...","Saint George, BM",1990-1997,A large cluster of oceanographic datasets from...,0.591898,0.837477
3,3,3815,"[nathaniel, palmer, jgofs, ctd, revelle, roger...","Tasmania, AU",1997-2000,A large cluster of oceanographic and marine re...,0.645258,0.843684
4,4,3755,"[censor, bottle, chile, odp, physical, zooplan...","Ica, PE",1995-2004,A large cluster of datasets from roughly 1995–...,0.676345,0.820739
5,5,3289,"[meteor, geob, sediment, type, core, corer, gr...","Karas, NA",1992-2000,A large cluster of datasets from 1992–2000 is ...,0.698308,0.817049
6,6,23952,"[bebauungsplan, inspireidentifiziert, bodennut...","North Rhine-Westphalia, DE",1993-2023,A large cluster of datasets from around North ...,0.710504,0.800978
7,7,4365,"[radiosonde, ant, polarstern, cruise, xxix, zo...",", GS",1995-2014,This cluster appears to cover 1995–2014 oceano...,0.515756,0.776444
8,8,5783,"[ice, ant, neumayer, polarstern, antarctic, an...",", FK",1992-2018,A large collection of datasets from 1992–2018 ...,0.668194,0.845307
9,9,2092,"[queen, ark, arctic, awi, land, expedition, se...","Sakha, RU",1993-2014,A large set of datasets from roughly 1993–2014...,0.667301,0.844574


In [ ]:
explainer_gpt5.save_explanations(path=expl_dir_gpt5)

Saving explanations...
Saving complete.


In [ ]:
expl_dir_gpt5_nano = PROCESSED_DATA_DIR + '/Explanations/gpt-5.4-nano.xlsx'
explainer_gpt5_nano = ClusterExplainer(df)
explainer_gpt5_nano.fit_explanations(top_n=15)
explainer_gpt5_nano.generate_explainations(model_name='gpt-5.4-nano')
explainer_gpt5_nano_eval = explainer_gpt5_nano.evaluate_results()
explainer_gpt5_nano_eval

Extracting keywords and location...
Extraction complete.
Generating explanations using gpt-5.4-nano...
Generation complete.
Method generate_explainations took 21.6810s
Evaluating...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluation complete.


,cluster_id,size,keywords,region,period,explanation,kw_fidelity,exp_faithfulness
0,0,2110,"[bsrn, station, monitoring, atmosphere, violet...","Oklahoma, US",1997-2017,This cluster consists of 2110 datasets from Ok...,0.642745,0.829150
1,1,2857,"[dendrochronological, object, historical, tree...","Overijssel, NL",2003-2011,This cluster collects studies from around Over...,0.606389,0.823296
2,2,2479,"[xbt, profiles, woce, oceanography, cruise, wa...","Saint George, BM",1990-1997,This cluster focuses on 1990–1997 oceanographi...,0.591898,0.837582
3,3,3815,"[nathaniel, palmer, jgofs, ctd, revelle, roger...","Tasmania, AU",1997-2000,This cluster contains datasets about marine sc...,0.645258,0.835874
4,4,3755,"[censor, bottle, chile, odp, physical, zooplan...","Ica, PE",1995-2004,This cluster contains 1995–2004 oceanographic ...,0.676345,0.827409
5,5,3289,"[meteor, geob, sediment, type, core, corer, gr...","Karas, NA",1992-2000,This cluster centers on sediment and gravity-r...,0.698308,0.826748
6,6,23952,"[bebauungsplan, inspireidentifiziert, bodennut...","North Rhine-Westphalia, DE",1993-2023,This cluster is about local land-use and munic...,0.710504,0.801685
7,7,4365,"[radiosonde, ant, polarstern, cruise, xxix, zo...",", GS",1995-2014,"This cluster of 4,365 datasets focuses on pola...",0.515756,0.794127
8,8,5783,"[ice, ant, neumayer, polarstern, antarctic, an...",", FK",1992-2018,"This cluster of 5,783 datasets from roughly th...",0.668194,0.826902
9,9,2092,"[queen, ark, arctic, awi, land, expedition, se...","Sakha, RU",1993-2014,This cluster summarizes Arctic research on per...,0.667301,0.832403


In [ ]:
explainer_gpt5_nano.save_explanations(path=expl_dir_gpt5_nano)

Saving explanations...
Saving complete.


In [ ]:
expl_dir_4o_mini = PROCESSED_DATA_DIR + '/Explanations/gpt-4o-mini.xlsx'
explainer_4o_mini = ClusterExplainer(df)
explainer_4o_mini.fit_explanations(top_n=15)
explainer_4o_mini.generate_explainations(model_name='gpt-4o-mini')
explainer_4o_mini_eval = explainer_4o_mini.evaluate_results()
explainer_4o_mini_eval

Extracting keywords and location...
Extraction complete.
Generating explanations using gpt-4o-mini...
Generation complete.
Method generate_explainations took 19.7688s
Evaluating...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluation complete.


,cluster_id,size,keywords,region,period,explanation,kw_fidelity,exp_faithfulness
0,0,2110,"[bsrn, station, monitoring, atmosphere, violet...","Oklahoma, US",1997-2017,"The cluster comprises 2,110 datasets focused o...",0.642745,0.830464
1,1,2857,"[dendrochronological, object, historical, tree...","Overijssel, NL",2003-2011,"The cluster comprises 2,857 datasets from the ...",0.606389,0.824067
2,2,2479,"[xbt, profiles, woce, oceanography, cruise, wa...","Saint George, BM",1990-1997,"The cluster comprises 2,479 oceanographic data...",0.591898,0.828709
3,3,3815,"[nathaniel, palmer, jgofs, ctd, revelle, roger...","Tasmania, AU",1997-2000,"The cluster comprises 3,815 datasets related t...",0.645258,0.823139
4,4,3755,"[censor, bottle, chile, odp, physical, zooplan...","Ica, PE",1995-2004,"The cluster comprises 3,755 datasets related t...",0.676345,0.827091
5,5,3289,"[meteor, geob, sediment, type, core, corer, gr...","Karas, NA",1992-2000,"The cluster comprises 3,289 datasets focused o...",0.698308,0.825461
6,6,23952,"[bebauungsplan, inspireidentifiziert, bodennut...","North Rhine-Westphalia, DE",1993-2023,"The cluster comprises 23,952 datasets related ...",0.710504,0.793915
7,7,4365,"[radiosonde, ant, polarstern, cruise, xxix, zo...",", GS",1995-2014,"The cluster comprises 4,365 datasets centered ...",0.515756,0.776643
8,8,5783,"[ice, ant, neumayer, polarstern, antarctic, an...",", FK",1992-2018,"The cluster comprises 5,783 datasets focused o...",0.668194,0.835905
9,9,2092,"[queen, ark, arctic, awi, land, expedition, se...","Sakha, RU",1993-2014,"The cluster comprises 2,092 datasets focused o...",0.667301,0.828766


In [ ]:
explainer_4o_mini.save_explanations(path=expl_dir_4o_mini)

Saving explanations...
Saving complete.


In [ ]:
expl_dir_gpt4 = PROCESSED_DATA_DIR + '/Explanations/gpt-4.1.xlsx'
explainer_gpt4 = ClusterExplainer(df)
explainer_gpt4.fit_explanations(top_n=15)
explainer_gpt4.generate_explainations(model_name='gpt-4.1')
explainer_gpt4_eval = explainer_gpt4.evaluate_results()
explainer_gpt4_eval

Extracting keywords and location...
Extraction complete.
Generating explanations using gpt-4.1...
Generation complete.
Method generate_explainations took 27.0701s
Evaluating...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluation complete.


,cluster_id,size,keywords,region,period,explanation,kw_fidelity,exp_faithfulness
0,0,2110,"[bsrn, station, monitoring, atmosphere, violet...","Oklahoma, US",1997-2017,"This cluster consists of 2,110 datasets collec...",0.642745,0.833065
1,1,2857,"[dendrochronological, object, historical, tree...","Overijssel, NL",2003-2011,"This cluster consists of 2,857 datasets from a...",0.606389,0.825193
2,2,2479,"[xbt, profiles, woce, oceanography, cruise, wa...","Saint George, BM",1990-1997,"This cluster comprises 2,479 oceanographic dat...",0.591898,0.832015
3,3,3815,"[nathaniel, palmer, jgofs, ctd, revelle, roger...","Tasmania, AU",1997-2000,"This cluster comprises 3,815 datasets centered...",0.645258,0.834619
4,4,3755,"[censor, bottle, chile, odp, physical, zooplan...","Ica, PE",1995-2004,"This cluster comprises 3,755 datasets centered...",0.676345,0.826796
5,5,3289,"[meteor, geob, sediment, type, core, corer, gr...","Karas, NA",1992-2000,"This cluster, centered around the Karas region...",0.698308,0.819057
6,6,23952,"[bebauungsplan, inspireidentifiziert, bodennut...","North Rhine-Westphalia, DE",1993-2023,"This cluster contains 23,952 datasets from aro...",0.710504,0.803101
7,7,4365,"[radiosonde, ant, polarstern, cruise, xxix, zo...",", GS",1995-2014,"This cluster consists of 4,365 datasets from a...",0.515756,0.773322
8,8,5783,"[ice, ant, neumayer, polarstern, antarctic, an...",", FK",1992-2018,"This cluster contains 5,783 datasets collected...",0.668194,0.836056
9,9,2092,"[queen, ark, arctic, awi, land, expedition, se...","Sakha, RU",1993-2014,"This cluster contains 2,092 datasets from arou...",0.667301,0.830583


In [ ]:
explainer_gpt4.save_explanations(path=expl_dir_gpt4)

Saving explanations...
Saving complete.


In [ ]:
expl_dir_gpt5_pro = PROCESSED_DATA_DIR + '/Explanations/gpt-5.4-pro.xlsx'
explainer_gpt5_pro = ClusterExplainer(df)
explainer_gpt5_pro.fit_explanations(top_n=15)
explainer_gpt5_pro.generate_explainations(model_name='gpt-5.4-pro')
explainer_gpt5_pro_eval = explainer_gpt5_pro.evaluate_results()
explainer_gpt5_pro_eval

Extracting keywords and location...
Extraction complete.
Generating explanations using gpt-5.4-pro...


KeyboardInterrupt: 

In [ ]:
explainer_gpt5_pro.save_explanations(path=expl_dir_gpt5_pro)